In [2]:
%use intellij-platform
loadPlugins("com.intellij.java")

In [8]:
import com.intellij.psi.PsiElement
import com.intellij.psi.PsiMethod
import com.intellij.psi.PsiReference
import com.intellij.psi.javadoc.CustomJavadocTagProvider
import com.intellij.psi.javadoc.JavadocTagInfo
import com.intellij.psi.javadoc.PsiDocTagValue
import org.jetbrains.annotations.Nls

registerProjectExtension(JavadocTagInfo.EP_NAME.name, object : JavadocTagInfo {
    override fun getName() = "luaReturn"

    override fun isInline() = false

    override fun isValidInContext(context: PsiElement?) = context is PsiMethod

    override fun checkTagValue(value: PsiDocTagValue?) = null

    override fun getReference(p0: PsiDocTagValue?) = null
})

registerProjectExtension(JavadocTagInfo.EP_NAME.name, object : JavadocTagInfo {
    override fun getName() = "luaParam"

    override fun isInline() = false

    override fun isValidInContext(context: PsiElement?) = context is PsiMethod

    override fun checkTagValue(value: PsiDocTagValue?) = null

    override fun getReference(p0: PsiDocTagValue?) = null
})

In [4]:
import com.intellij.execution.configurations.GeneralCommandLine
import com.intellij.execution.process.CapturingProcessAdapter
import com.intellij.execution.process.OSProcessHandler
import com.intellij.execution.process.ProcessEvent
import com.intellij.formatting.service.AsyncDocumentFormattingService
import com.intellij.formatting.service.AsyncFormattingRequest
import com.intellij.formatting.service.FormattingService
import com.intellij.openapi.util.NlsSafe
import com.intellij.psi.PsiFile
import java.nio.charset.StandardCharsets
import java.util.EnumSet

class StyluaFormatter : AsyncDocumentFormattingService() {

    override fun getName() = "StyLua Formatter"

    override fun getNotificationGroupId() = "StyLua Formatter"

    override fun getFeatures() = EnumSet.noneOf(FormattingService.Feature::class.java)

    override fun canFormat(file: PsiFile): Boolean {
        return file.virtualFile.extension == "luau"
    }

    override fun createFormattingTask(request: AsyncFormattingRequest): FormattingTask? {
        val path = request.ioFile?.toPath() ?: return null

        val commandLine = GeneralCommandLine()
            .withParentEnvironmentType(GeneralCommandLine.ParentEnvironmentType.CONSOLE)
            .withExePath("stylua")
            .withWorkingDirectory(path.parent)
            .withParameters("--no-editorconfig", "-")
            .withInput(request.ioFile)
        val handler = OSProcessHandler(commandLine.withCharset(StandardCharsets.UTF_8))
        return object : FormattingTask {
            override fun run() {
                handler.addProcessListener(object : CapturingProcessAdapter() {
                    override fun processTerminated(event: ProcessEvent) {
                        if (event.exitCode == 0) {
                            request.onTextReady(output.stdout)
                        } else {
                            request.onTextReady(output.stderr)
                        }
                    }
                })
                handler.startNotify()
            }

            override fun cancel(): Boolean {
                handler.destroyProcess()
                return true
            }

            override fun isRunUnderProgress(): Boolean {
                return true
            }
        }
    }
}

registerExtension(FormattingService.EP_NAME, StyluaFormatter())